#### Reading the instances

In [1]:
# Step 1: Read the text file and store its lines in a list.
name = 'ct_6_n_3_k10_t_60.txt'
with open(name, 'r') as f:
    lines = f.readlines()

# Step 2: Extract necessary information from the list.
num_vehicles = None
vehicle_capacity = None
demands = []
locations = []

for i, line in enumerate(lines):
    if "Min no of trucks" in line:
        num_vehicles = int(line.split("Min no of trucks:")[1].split(",")[0].strip())
    elif "CAPACITY" in line:
        vehicle_capacity = int(line.split(":")[1].strip())
    elif "NODE_COORD_SECTION" in line:
        j = i + 1
        while lines[j].strip() != "DEMAND_SECTION":
            _, x, y = map(float, lines[j].split())
            locations.append((x, y))
            j += 1
    elif "DEMAND_SECTION" in line:
        j = i + 1
        while lines[j].strip() != "DEPOT_SECTION":
            _, demand = map(float, lines[j].split())
            demands.append(demand)
            j += 1

# Removing the depot's demand, which is always 0
demands.pop(0)
num_customers = len(demands)

# Step 3: Use the extracted information to generate the desired Python output.
print("# Given data")
print(f"num_vehicles = {num_vehicles}")
print(f"num_customers = {num_customers}")
print(f"vehicle_capacity = {vehicle_capacity}")
print(f"demands = {demands}")
print("\nlocations = [")
for i, (x, y) in enumerate(locations):
    if i == 0:
        print(f"    ({x}, {y}),  # depot")
    else:
        print(f"    ({x}, {y}),  # customer {i}")
print("]")

# Given data
num_vehicles = 3
num_customers = 6
vehicle_capacity = 10
demands = [3.0, 3.8, 2.3, 2.0, 2.8, 4.1]

locations = [
    (0.0, 0.0),  # depot
    (2.0, 4.0),  # customer 1
    (8.0, 1.0),  # customer 2
    (-1.0, -10.0),  # customer 3
    (-5.0, -6.0),  # customer 4
    (-7.0, 5.0),  # customer 5
    (10.0, -6.0),  # customer 6
]


In [2]:
import numpy as np
def generate_intr_matrix(n, a, b):
    # Generate the upper triangle of the matrix with random numbers between min and max
    upper_triangle = np.triu(np.random.uniform(a, b, (n, n)), 1)

        # Create the symmetric matrix by adding the transpose of the upper triangle to itself and the diagonal as 1
    matrix = upper_triangle + upper_triangle.T + np.eye(n)

    # Convert numpy array to a list of lists
    interdiction_probabilities = matrix.tolist()
    return interdiction_probabilities


min = 1.0
max = 1.0
n = 7
interdiction_probabilities_1 = generate_intr_matrix(n, min, max)
interdiction_probabilities_1

[[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]]

In [2]:
interdiction_probabilities_0 = [[1.0,
  0.9586622174873944,
  0.9036063173610683,
  0.9832040990167482,
  0.9076118802278522,
  0.9394986349868428,
  0.926734549983961],
 [0.9586622174873944,
  1.0,
  0.9692412413165818,
  0.9844024972559998,
  0.9306483086619486,
  0.923515580417536,
  0.948950846106156],
 [0.9036063173610683,
  0.9692412413165818,
  1.0,
  0.9631104702275359,
  0.9219494612787523,
  0.9860629148689243,
  0.9046775821583093],
 [0.9832040990167482,
  0.9844024972559998,
  0.9631104702275359,
  1.0,
  0.9183008612950844,
  0.9606346614818988,
  0.9060878041801781],
 [0.9076118802278522,
  0.9306483086619486,
  0.9219494612787523,
  0.9183008612950844,
  1.0,
  0.9662425058707746,
  0.9664425079392789],
 [0.9394986349868428,
  0.923515580417536,
  0.9860629148689243,
  0.9606346614818988,
  0.9662425058707746,
  1.0,
  0.9525353543425883],
 [0.926734549983961,
  0.948950846106156,
  0.9046775821583093,
  0.9060878041801781,
  0.9664425079392789,
  0.9525353543425883,
  1.0]]


#### Formulation Not visiting Customers with demand 0 to be serviced

Eliminating cosntraint that if path is selected, it must have at least 1.0 demand serviced

Adding cumulative probabilities and all paths (no symmetry when there are interdictions)

In [3]:
from itertools import chain, combinations, permutations
from docplex.mp.model import Model
import time

matrx = '0'
if matrx == '0':
    interdiction_probabilities = interdiction_probabilities_0
else:
    interdiction_probabilities = interdiction_probabilities_1

# Given data
max_time = 7200

start_time = time.time()
# Generate a distance matrix based on Euclidean distance
distance_matrix = [[(((x1-x2)**2 + (y1-y2)**2)**0.5) for x2, y2 in locations] for x1, y1 in locations]

def is_matrix_uniform(inter_prob_matrix):
    """Check if the interdiction matrix has all elements as 1."""
    for row in inter_prob_matrix:
        for prob in row:
            if prob != 1:
                return False
    return True

uniform_matrix = is_matrix_uniform(interdiction_probabilities)

# Create the model
mdl = Model("SDVRP")

# Generate all feasible paths considering symmetry
def generate_all_paths():
    def is_unique_path(path, existing_paths_set):
        # Check if path or its reverse is not already in existing_paths_set
        return tuple(path) not in existing_paths_set and tuple(path[::-1]) not in existing_paths_set

    unique_paths_set = set()
    all_paths = []

    for length in range(1, num_customers + 1):  # Length of customer subset
        for subset in combinations(range(1, num_customers + 1), length):
            for perm in permutations(subset):
                path = (0,) + perm + (0,)
                if is_unique_path(path, unique_paths_set):
                    unique_paths_set.add(tuple(path))  # Add to set for quick lookup
                    all_paths.append(path)  # Add to list for output

    return all_paths

def generate_all_paths_no_symmetry(num_customers):
    """Generate all paths without considering symmetry, for any number of customers."""
    all_paths = []
    # Generate paths for all possible subsets of customers (including subsets of size 1 to num_customers)
    for r in range(1, num_customers + 1):
        for subset in combinations(range(1, num_customers + 1), r):
            for perm in permutations(subset):
                path = (0,) + perm + (0,)  # Adding depot at the start and end
                all_paths.append(path)
    return all_paths

# Generate all feasible paths
#def generate_all_paths():
#    all_paths = []
#    for length in range(2, num_customers + 2):
#        for subset in combinations(range(1, num_customers + 1), length - 2):
#            all_paths.append((0,) + subset + (0,))
#    return all_paths


#all_paths = generate_all_paths()

# Assuming num_customers is defined
if uniform_matrix:
    all_paths = generate_all_paths()  # Use your existing symmetry-considering function
else:
    all_paths = generate_all_paths_no_symmetry(num_customers)


print('all paths were generated')
print('elapsed time: ', time.time()-start_time)

# Function to calculate cumulative interdiction probabilities for each customer in a path
def calculate_cumulative_probability(path, interdiction_probabilities):
    cumulative_probabilities = {path[0]: 1}  # Start with depot having a cumulative probability of 1
    for i in range(1, len(path)):
        cumulative_probabilities[path[i]] = cumulative_probabilities[path[i-1]] * interdiction_probabilities[path[i-1]][path[i]]
    return cumulative_probabilities

print('all cumulative probabilities for all paths were generated')
print('elapsed time: ', time.time()-start_time)

# Define binary variables for each path and continuous variables for the demand delivered on each path to a customer
path_vars = mdl.binary_var_dict(all_paths, name="path")

print('all paths vars were generated')
print('elapsed time: ', time.time()-start_time)

deliver_vars = mdl.continuous_var_matrix(all_paths, range(1, num_customers + 1), lb=0, name="deliver")

print('all deliver vars were generated')
print('elapsed time: ', time.time()-start_time)

# Objective: Minimize total distance traveled
mdl.minimize(mdl.sum(path_vars[p] * sum(distance_matrix[p[i]][p[i+1]] for i in range(len(p)-1)) for p in all_paths))

# Constraints
#for customer in range(1, num_customers + 1):
#    mdl.add_constraint(mdl.sum(deliver_vars[p, customer] for p in all_paths if customer in p) == demands[customer-1])

#for customer in range(1, num_customers + 1):
#    # This constraint considers interdiction probabilities for routes starting and ending at the depot
#    mdl.add_constraint(
#        mdl.sum(
#            interdiction_probabilities[0][customer] * deliver_vars[p, customer] if p[1] == customer and len(p) == 3
#            else interdiction_probabilities[customer][p[customer_pos+1]] * deliver_vars[p, customer]
#            for p in all_paths if customer in p
#            for customer_pos in range(1, len(p) - 1) if p[customer_pos] == customer
#        ) >= demands[customer-1],
#        "demand_satisfaction_%d" % customer
#    )

# After defining 'all_paths', 'path_vars', and 'deliver_vars'
for customer in range(1, num_customers + 1):
    mdl.add_constraint(
        mdl.sum(
            calculate_cumulative_probability(p, interdiction_probabilities)[customer] * deliver_vars[p, customer]
            for p in all_paths if customer in p[1:-1]  # Ensure customer is in the path (excluding depot)
        ) >= demands[customer-1],
        f"demand_satisfaction_with_interdiction_{customer}"
    )



print('1st part of constraints were generated')
print('elapsed time: ', time.time()-start_time)

for p in all_paths:
    #for customer in p:
        #if customer != 0:  # exclude depot
            #mdl.add_constraint(deliver_vars[p, customer] <= demands[customer-1] * path_vars[p])
            #mdl.add_constraint(deliver_vars[p, customer] >= path_vars[p])
    mdl.add_constraint(mdl.sum(deliver_vars[p, customer] for customer in p if customer != 0) <= vehicle_capacity * path_vars[p])

for p in all_paths:
    for customer in range(1, num_customers + 1):
        if customer in p:  # Only apply if the customer is in the path
            # This constraint ensures demand serviced to a customer in a route is less or equal to vehicle capacity if route is selected
            mdl.add_constraint(deliver_vars[p, customer] <= vehicle_capacity * path_vars[p], f"vehicle_cap_{p}_{customer}")

#mdl.add_constraint(mdl.sum(path_vars[p] for p in all_paths) <= num_customers)  # The number of vehicles is equal to the number of customers in this case
mdl.add_constraint(mdl.sum(path_vars[p] for p in all_paths) <= num_vehicles)  # The number of vehicles is equal to the number of customers in this case


print('all paths constraints were generated')
print('elapsed time: ', time.time()-start_time)

# Solve the model
mdl.parameters.timelimit = max_time
mdl.parameters.mip.display = 4
solution = mdl.solve(log_output=True)

end_time = time.time()
elapsed_time = end_time - start_time

# Print solution
if solution:
    print(f"Solution Status: {mdl.solve_details.status}")
    print(f"Total cost (distance traveled): {solution.objective_value}")
    total_demand_serviced = sum(demands)
    print(f"Total demand serviced: {total_demand_serviced}")
    print()

    print("\nConstraint Details with Cumulative Interdiction Probabilities and Deliveries:")
    for customer in range(1, num_customers + 1):
        print(f"\nCustomer {customer} demand: {demands[customer-1]}")
        for p in all_paths:
            if customer in p:
                # Calculate cumulative probabilities for this path
                cumulative_probabilities = calculate_cumulative_probability(p, interdiction_probabilities)
                customer_pos = p.index(customer)
                # Determine if the customer is directly after the depot in the route
                if customer_pos == 1:  # Customer is the first after the depot
                    prev_node = 0
                else:  # Customer is not the first, find the previous node
                    prev_node = p[customer_pos - 1]
                
                interdiction_probability = cumulative_probabilities[customer]
                delivery_amount = deliver_vars[p, customer].solution_value if solution else 0
                adjusted_delivery = interdiction_probability * delivery_amount

                # Print the path, cumulative interdiction probability, and adjusted delivery amount
                print(f"  Path {p}: Cumulative Interdiction Probability to Customer {customer}: {interdiction_probability}")
                print(f"    Expected delivery amount (adjusted for interdiction): {adjusted_delivery}")


   
    
    route_number = 1
    for p, var in path_vars.items():
        if var.solution_value > 0.5:
            route_cost = sum(distance_matrix[p[i]][p[i+1]] for i in range(len(p)-1))
            total_demand_route = sum(deliver_vars[p, customer].solution_value for customer in p if customer != 0)
            print(f"Route {route_number}: {p}")
            print(f"  Cost: {route_cost}")
            print(f"  Total demand serviced by route: {total_demand_route}")
            for customer in p:
                if customer != 0:  # exclude depot
                    print(f"    Customer {customer} demand serviced: {deliver_vars[p, customer].solution_value}")
            route_number += 1
    
    print()
    
    split_customers = [c for c in range(1, num_customers + 1) if sum(path_vars[p].solution_value for p in all_paths if c in p and deliver_vars[p, c].solution_value > 0) > 1]
    if split_customers:
        print("Customers with split delivery:")
        for c in split_customers:
            print(f"  Customer {c}:")
            for p, var in path_vars.items():
                if c in p and var.solution_value > 0.5:
                    print(f"    Demand serviced by route {p}: {deliver_vars[p, c].solution_value}")
    else:
        print("No customers have split delivery.")
else:
    print("No solution found.")

print(f"Elapsed time: {elapsed_time} seconds")

all paths were generated
elapsed time:  0.019517183303833008
all cumulative probabilities for all paths were generated
elapsed time:  0.019517183303833008
all paths vars were generated
elapsed time:  0.030518054962158203
all deliver vars were generated
elapsed time:  0.0826256275177002
1st part of constraints were generated
elapsed time:  0.16344451904296875
all paths constraints were generated
elapsed time:  0.7671217918395996
Version identifier: 22.1.1.0 | 2022-11-27 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
CPXPARAM_MIP_Display                             4
CPXPARAM_TimeLimit                               7200
Tried aggregator 2 times.
MIP Presolve eliminated 9786 rows and 1950 columns.
MIP Presolve modified 6 coefficients.
Aggregator did 6 substitutions.
Reduced MIP has 1957 rows, 11736 columns, and 23472 nonzeros.
Reduced MIP has 1956 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.03 sec. (27.68 ticks)
Tried aggregator 1 time.
Detecting symm